# Entrenamiento de mT5: chileno → español neutro

Ajusta `google/mt5-small` con `data/interim/borrador_entrenamiento.csv` y lo evalúa en los lotes revisados. Se basa en `idea_flo/entrenar_mt5.py`.

**Antes de ejecutar:** en Colab, elige *Entorno de ejecución → Cambiar tipo de entorno → GPU T4* y luego *Ejecutar todas*. Colab pedirá autorizar Google Drive.

**Se guarda en Google Drive:**

| Contenido | Ruta |
|---|---|
| Pesos, configuración y tokenizer | `/content/drive/MyDrive/proyecto-nlp/models/mt5_flo_fp32/` |
| Predicciones | `/content/drive/MyDrive/proyecto-nlp/data/processed/predicciones_mt5_flo_fp32.csv` |
| BLEU y chrF | `/content/drive/MyDrive/proyecto-nlp/resultados/comparacion_mt5.csv` |

Los pesos ocupan cerca de 1,2 GB. Durante el entrenamiento el mejor modelo queda en el disco de Colab y se copia a Drive una sola vez al terminar, para no llenar la papelera de Drive con versiones intermedias.

## 1. Configuración

In [ ]:
RAMA = "mt5-flo-archivos"  # rama de GitHub que trae los datos de entrenamiento
URL_REPO = "https://github.com/carlosiskindacool/proyecto-nlp.git"
MODELO = "google/mt5-small"

PRECISION = "fp32"  # "fp32" o "bf16". La T4 no tiene bf16 por hardware: usar fp32.
EPOCAS = 8
LOTE = 8
LR = 5e-4
VAL = 0.05
SEMILLA = 1
SOBRESCRIBIR = False  # True para reemplazar pesos ya guardados en Drive

ENTRENAMIENTO = "data/interim/borrador_entrenamiento.csv"
PARES = ["annotation/lote1.csv", "annotation/lote_1_noticias.csv", "annotation/lote_1_tuits.csv"]

if PRECISION not in ("bf16", "fp32"):
    raise ValueError('PRECISION debe ser "bf16" o "fp32"')

NOMBRE = f"mt5_flo_{PRECISION}"
RAIZ_DRIVE = "/content/drive"
CARPETA_DRIVE = f"{RAIZ_DRIVE}/MyDrive/proyecto-nlp"
DIR_PESOS = f"{CARPETA_DRIVE}/models/{NOMBRE}"
RUTA_PREDICCIONES = f"{CARPETA_DRIVE}/data/processed/predicciones_{NOMBRE}.csv"
RUTA_RESUMEN = f"{CARPETA_DRIVE}/resultados/comparacion_mt5.csv"
DIR_LOCAL = f"/content/{NOMBRE}_mejor"
REPO = "/content/proyecto-nlp"

## 2. Montar Google Drive

In [ ]:
import os
import shutil

from google.colab import drive

drive.mount(RAIZ_DRIVE)
if not os.path.isdir(f"{RAIZ_DRIVE}/MyDrive"):
    raise RuntimeError("Google Drive no quedó montado; los pesos se perderían al cerrar Colab.")

if os.path.exists(f"{DIR_PESOS}/model.safetensors") and not SOBRESCRIBIR:
    raise FileExistsError(f"Ya hay pesos en {DIR_PESOS}. Usa SOBRESCRIBIR = True o cambia NOMBRE.")

for carpeta in (os.path.dirname(DIR_PESOS), os.path.dirname(RUTA_PREDICCIONES), os.path.dirname(RUTA_RESUMEN)):
    os.makedirs(carpeta, exist_ok=True)

libre_gb = shutil.disk_usage(f"{RAIZ_DRIVE}/MyDrive").free / 1024**3
print(f"Drive montado | libre: {libre_gb:.1f} GB | pesos en: {DIR_PESOS}")
if libre_gb < 3:
    print("Aviso: conviene tener al menos 3 GB libres en Drive para copiar los pesos.")

## 3. Instalar dependencias y traer el repositorio

In [ ]:
!pip install -q "transformers==5.24.0" sentencepiece sacrebleu

In [ ]:
import subprocess
import sys

if os.path.isdir(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA, URL_REPO, REPO], check=True)

os.chdir(REPO)
sys.path.insert(0, f"{REPO}/src/chilenismos")

## 4. Preparar GPU, precisión y datos

In [ ]:
import csv
import json
import math
import random

import sacrebleu
import torch
from transformers import Adafactor, AutoModelForSeq2SeqLM, AutoTokenizer

from baselines import cargar_pares, guardar_resumen, limpiar

random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

dev = "cuda" if torch.cuda.is_available() else "cpu"
if dev != "cuda":
    raise RuntimeError("No hay GPU. Activa la T4 en Entorno de ejecución → Cambiar tipo de entorno.")

USAR_BF16 = PRECISION == "bf16" and dev == "cuda"
print("dispositivo:", dev, torch.cuda.get_device_name(0) if dev == "cuda" else "", "| precisión:", PRECISION)

In [ ]:
def cargar_entrenamiento(ruta, prohibidas):
    with open(ruta, encoding="utf-8") as f:
        filas = list(csv.DictReader(f, delimiter=";"))
    vistas, pares = set(), []
    for r in filas:
        o, n = limpiar(r["oracion"]), limpiar(r["neutro"])  # sin menciones ni emojis, igual que los conjuntos de prueba
        if not o or not n or n.upper() == "DESCARTAR" or o in prohibidas or o in vistas:
            continue
        vistas.add(o)
        pares.append((o, n))
    return pares


conjuntos = {ruta: cargar_pares(ruta) for ruta in PARES}
prohibidas = {p["chileno"] for ps in conjuntos.values() for p in ps}  # evita fuga hacia la evaluación
pares = cargar_entrenamiento(ENTRENAMIENTO, prohibidas)
random.shuffle(pares)
n_val = max(1, round(len(pares) * VAL))
val, train = pares[:n_val], pares[n_val:]
print(f"entrenamiento: {len(train)} pares | validación: {len(val)} | prueba: {sum(map(len, conjuntos.values()))}")

## 5. Entrenar

In [ ]:
tok = AutoTokenizer.from_pretrained(MODELO)
mod = AutoModelForSeq2SeqLM.from_pretrained(MODELO).to(dev)
opt = Adafactor(mod.parameters(), lr=LR, relative_step=False, scale_parameter=False, warmup_init=False)
mod.gradient_checkpointing_enable()
mod.config.use_cache = False


def codificar(lote):
    enc = tok([x[0] for x in lote], return_tensors="pt", padding=True, truncation=True, max_length=128).to(dev)
    lab = tok(text_target=[x[1] for x in lote], return_tensors="pt", padding=True, truncation=True,
              max_length=128).input_ids.to(dev)
    lab[lab == tok.pad_token_id] = -100
    return enc, lab


def guardar_mejor():
    mod.config.use_cache = True
    mod.save_pretrained(DIR_LOCAL)
    tok.save_pretrained(DIR_LOCAL)
    mod.config.use_cache = False


shutil.rmtree(DIR_LOCAL, ignore_errors=True)
mejor, mejor_epoca, historial = float("inf"), None, []
for ep in range(1, EPOCAS + 1):
    mod.train()
    random.shuffle(train)
    total = 0.0
    for i in range(0, len(train), LOTE):
        enc, lab = codificar(train[i:i + LOTE])
        with torch.autocast(dev, dtype=torch.bfloat16, enabled=USAR_BF16):
            loss = mod(**enc, labels=lab).loss
        loss.backward()
        opt.step()
        opt.zero_grad(set_to_none=True)
        total += loss.item()
        if (i // LOTE) % 100 == 0:
            print(f"  época {ep} paso {i // LOTE}/{len(train) // LOTE} pérdida {loss.item():.3f}", flush=True)

    mod.eval()
    perd = 0.0
    with torch.no_grad():
        for i in range(0, len(val), LOTE):
            b = val[i:i + LOTE]
            enc, lab = codificar(b)
            with torch.autocast(dev, dtype=torch.bfloat16, enabled=USAR_BF16):
                perd += mod(**enc, labels=lab).loss.item() * len(b)
    perd /= len(val)
    perd_train = total / max(1, math.ceil(len(train) / LOTE))
    historial.append({"epoca": ep, "perdida_entrenamiento": perd_train, "perdida_validacion": perd})
    print(f"época {ep}: pérdida de entrenamiento {perd_train:.3f} | validación {perd:.3f}", flush=True)
    if perd < mejor:
        mejor, mejor_epoca = perd, ep
        guardar_mejor()

if mejor_epoca is None:
    raise RuntimeError("La pérdida de validación no fue válida en ninguna época; no se guardó el modelo.")
print(f"Mejor época: {mejor_epoca} (validación {mejor:.3f})")

## 6. Guardar los pesos en Drive

In [ ]:
with open(f"{DIR_LOCAL}/entrenamiento.json", "w", encoding="utf-8") as f:
    json.dump({"modelo": MODELO, "precision": PRECISION, "epocas": EPOCAS, "lote": LOTE, "lr": LR, "val": VAL,
               "semilla": SEMILLA, "mejor_epoca": mejor_epoca, "historial": historial}, f, ensure_ascii=False, indent=2)

temporal = f"{DIR_PESOS}.tmp"
shutil.rmtree(temporal, ignore_errors=True)
shutil.copytree(DIR_LOCAL, temporal)
if os.path.isdir(DIR_PESOS):
    shutil.rmtree(DIR_PESOS)
os.rename(temporal, DIR_PESOS)

pesos = f"{DIR_PESOS}/model.safetensors"
if os.path.getsize(pesos) != os.path.getsize(f"{DIR_LOCAL}/model.safetensors"):
    raise RuntimeError("La copia de los pesos en Drive quedó incompleta.")
print(f"Pesos guardados en {DIR_PESOS} ({os.path.getsize(pesos) / 1024**3:.2f} GB)")

## 7. Evaluar el modelo guardado

In [ ]:
def traducir(modelo, frases, lote=16):
    salidas = []
    modelo.eval()
    for i in range(0, len(frases), lote):
        enc = tok(frases[i:i + lote], return_tensors="pt", padding=True, truncation=True, max_length=128).to(dev)
        with torch.no_grad(), torch.autocast(dev, dtype=torch.bfloat16, enabled=USAR_BF16):
            out = modelo.generate(**enc, max_new_tokens=96, num_beams=4)
        salidas += tok.batch_decode(out, skip_special_tokens=True)
    return salidas


del mod, opt
torch.cuda.empty_cache()
tok = AutoTokenizer.from_pretrained(DIR_PESOS)
modelo_final = AutoModelForSeq2SeqLM.from_pretrained(DIR_PESOS).to(dev)
modelo_final.config.use_cache = True

filas, salidas = [], []
for ruta, ps in conjuntos.items():
    hip = traducir(modelo_final, [p["chileno"] for p in ps])
    ref = [p["neutro"] for p in ps]
    salidas += [(os.path.basename(ruta), p["chileno"], r, p["termino"], h) for p, r, h in zip(ps, ref, hip)]
    bleu = sacrebleu.corpus_bleu(hip, [ref]).score
    chrf = sacrebleu.corpus_chrf(hip, [ref]).score
    print(f"{os.path.basename(ruta):<24} {NOMBRE:<18} BLEU {bleu:5.1f} chrF {chrf:5.1f}")
    filas.append({"sistema": NOMBRE, "escenario": "", "bleu": f"{bleu:.1f}", "chrf": f"{chrf:.1f}", "n_pares": len(ps),
                  "pares": ruta, "nota": f"{MODELO}; {PRECISION}; ajustado con borradores del LLM"})

with open(RUTA_PREDICCIONES, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f)
    w.writerow(["conjunto", "chileno", "referencia", "termino", NOMBRE])
    w.writerows(salidas)
guardar_resumen(RUTA_RESUMEN, filas)

## 8. Confirmar la escritura en Drive

In [ ]:
print("Pesos:", DIR_PESOS)
print("Predicciones:", RUTA_PREDICCIONES)
print("Resumen:", RUTA_RESUMEN)
drive.flush_and_unmount()
print("Drive sincronizado y desmontado.")